In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# load modules related to this exercise
from model import model_bufferstock

# Exercise 3: Solving the buffer-stock consumption model with EGM

Consider the canonical buffer-stock consumption model. You will complete the EGM step in `egm.py`, then use `model.py` to solve and simulate the life-cycle model.


Bellman equation in ratio form: 

$$\begin{align*}
v_{t}(m_{t}) & = & \max_{c_{t}}\left\{ \frac{c_{t}^{1-\rho}}{1-\rho}+\beta\mathbb{E}_{t}\left[(G L_{t} \psi_{t+1})^{1-\rho}v_{t+1}(m_{t+1})\right]\right\} \\
 & \text{s.t.}\\
 a_t & = & m_t-c_t \\
m_{t+1} & = & \frac{1}{G L_t \psi_{t+1}}Ra_t+\xi_{t+1}\\ 
a_t & \geq & \max(-\lambda_t,-\Omega_t) \\
\lambda_t & = & 
\begin{cases} 
\lambda & t < T_R\\
0 & t \geq T_R
\end{cases} \\
\Omega_t & = & \text{Maximum guaranteed repayable debt at time t} \\

\text{For } t<T_R: \\
\psi_{t+1} & \sim & \exp \mathcal{N}(-0.5 \sigma^2_{\psi},\sigma^2_{\psi})\\
\xi_{t+1}& = & \begin{cases}
\mu  &\text{with prob. }\pi\\
(\epsilon_{t+1}-\pi \mu)/ (1-\pi) &\text{else}
\end{cases}\\ \\
\epsilon_{t+1} & \sim & \exp \mathcal{N}(-0.5 \sigma^2_{\xi},\sigma^2_{\xi}) \\

\text{For } t\ge T_R: \\
\psi_{t+1} & = 1 \\
\xi_{t+1} & = 1 \\
\end{align*}$$

and remember:
$$\begin{align*}
Y_{t+1}& = & \xi_{t+1} P_{t+1} \\
P_{t+1}& = & GL_tP_t\psi_{t+1}\\ 
c_t & \equiv & C_t/P_t \\
m_t & \equiv & M_t/P_t \\
a_t & \equiv & A_t/P_t \\
p_t & \equiv & \ln(P_t) \\
y_t & \equiv & \ln(Y_t) \\
\end{align*}$$

The Euler equation is given by:
$$\begin{align*}
c_t^{-\rho} & = & \beta R \mathbb{E}_{t}\left[(G L_{t} \psi_{t+1})^{-\rho} c_{t+1}^{-\rho}\right] \\
\end{align*}$$


## Baseline calibration and parameters

The model distinguishes preference parameters, income risk, and numerical settings. The most useful parameters for economic experiments are:

| Concept | Code | Baseline | Interpretation |
|---|---:|---:|---|
| Discount factor | `beta` | 0.96 | Higher values make households more patient. |
| Relative risk aversion | `rho` | 2.0 | Also implies an intertemporal elasticity of substitution of $1/\rho$. |
| Gross interest factor | `R` | 1.04 | Return on end-of-period assets. |
| Transitory income risk | `sigma_xi` | 0.10 | Dispersion of temporary income shocks. |
| Permanent income risk | `sigma_psi` | 0.10 | Dispersion of permanent-income innovations. |
| Low-income probability | `low_p` | 0.005 | Probability of the discrete low-income event. |
| Borrowing limit | `lambdaa` | 0.0 | Maximum exogenous debt; zero means no borrowing. |

Parameters should be changed after `life_cycle_setup()` and before `create_grids()`, because the latter constructs shock nodes and state grids from the calibration.


## 1. Read [`README.md`](README.md) for the notebook roadmap, file map, parameter workflow, and numerical interpretation.


## 2. Ensure you understand the following functions:
<ol type="a">
<li> model.setup </li>
<li> model.create_grids </li>
<li> model.solve</li>
</ol>

## 3. Complete the missing EGM calculations in `egm.EGM_loop`.


In [ ]:
# load baseline settings
model = model_bufferstock()

In [ ]:
model.life_cycle_setup()

# solve and simulate
model.create_grids()
model.solve()
model.simulate()

par = model.par
sol = model.sol
sim = model.sim

## 4. Solve and simulate the baseline model, then inspect its policy and life-cycle profiles.


In [ ]:
fig, ax = plt.subplots(figsize=(8,5))
for age in [26, 35, 45, 55, 65, 75, par.T+par.age_min-1,par.T+par.age_min] :
    ax.plot(sol.m[age-par.age_min-1,:],sol.c[age-par.age_min-1,:], label=f'age = {age}')
ax.set_xlabel(f"$m_t$")
ax.set_ylabel(f"$c(m_t)$")
ax.set_xlim([np.min(par.a_min), 5])
ax.set_ylim([0,5])
ax.set_title(f'Consumption function')
ax.legend()

In [ ]:
fig, ax = plt.subplots(figsize=(8,5))
ax.plot(np.arange(par.simT)+par.age_min+1,np.mean(sim.Y,1))
ax.set_xlabel(f"age")
ax.set_ylabel(f"Income $Y_t$")
ax.set_title(f'Average income')

In [ ]:
fig, ax = plt.subplots(figsize=(8,5))
ax.plot(np.arange(par.simT)+par.age_min+1,np.mean(sim.M,1))
ax.set_xlabel(f"age")
ax.set_ylabel(f"Cash-on-hand $M_t$")
ax.set_title(f'Average Cash on hands')

In [ ]:
fig, ax = plt.subplots(figsize=(8,5))
ax.plot(np.arange(par.simT)+par.age_min+1,np.mean(sim.C,1))
ax.set_xlabel(f"age")
ax.set_ylabel(f"Consumption $C_t$")
ax.set_title(f'Average consumption')

In [ ]:
fig, ax = plt.subplots(figsize=(8,5))
ax.plot(np.arange(par.simT)+par.age_min+1,np.mean(sim.A,1))
ax.set_xlabel(f"age")
ax.set_ylabel(f"Asset $A_t$")
ax.set_title(f'Average Asset')

## 5. Optional: vectorize the EGM step

Replace the loop over end-of-period assets with array operations in `egm.EGM_vec`. Verify both correctness and speed: the vectorized and loop implementations should produce nearly identical policies, but the vectorized solver should be faster.


In [ ]:
import time

# Loop implementation
model_loop = model_bufferstock()
model_loop.life_cycle_setup()
model_loop.par.vec = False
model_loop.create_grids()
t0 = time.perf_counter()
model_loop.solve()
loop_time = time.perf_counter() - t0

# Vectorized implementation
model_vec = model_bufferstock()
model_vec.life_cycle_setup()
model_vec.par.vec = True
model_vec.create_grids()
t0 = time.perf_counter()
model_vec.solve()
vec_time = time.perf_counter() - t0

max_policy_difference = np.nanmax(np.abs(model_loop.sol.c - model_vec.sol.c))
print(f"Loop solution time:       {loop_time:.4f} seconds")
print(f"Vectorized solution time: {vec_time:.4f} seconds")
print(f"Speed-up:                 {loop_time / vec_time:.1f}x")
print(f"Maximum policy difference: {max_policy_difference:.3e}")

# Keep the vectorized objects available for the following plot.
par = model_vec.par
sol = model_vec.sol


In [ ]:
fig, ax = plt.subplots(figsize=(8,5))
for age in [26, 35, 45, 55, 65, 75, par.T+par.age_min-1,par.T+par.age_min] :
    ax.plot(sol.m[age-par.age_min-1,:],sol.c[age-par.age_min-1,:], label=f'age = {age}')
ax.set_xlabel(f"$m_t$")
ax.set_ylabel(f"$c(m_t)$")
ax.set_xlim([np.min(par.a_min), 5])
ax.set_ylim([0,5])
ax.set_title(f'Consumption function')
ax.legend()

## 6. Optional parameter laboratory: economic comparative statics

The baseline figures show one calibration. The experiments below change one primitive at a time while holding everything else fixed and using common random numbers.

1. **Patience:** How should a higher $\beta$ affect current consumption and life-cycle asset accumulation?
2. **Risk aversion / intertemporal substitution:** Increasing $\rho$ raises risk aversion but lowers the intertemporal elasticity of substitution, $1/\rho$. Which force appears strongest in the policy and asset profiles?
3. **Permanent-income risk:** How should a higher $\sigma_\psi$ affect precautionary saving? How might the answer differ for transitory risk, $\sigma_\xi$?

The reduced simulation size keeps this exploratory section quick. Edit the parameter lists and rerun the cell to formulate your own experiments.


In [ ]:
def solve_buffer_case(parameter, value, simN=10_000, policy_age=45):
    """Solve and simulate one comparative-static case using common random numbers."""
    case = model_bufferstock()
    case.life_cycle_setup()
    setattr(case.par, parameter, value)
    case.par.simN = simN
    case.create_grids()  # also resets the random seed to 2026
    case.solve()
    case.simulate()

    t = policy_age - case.par.age_min - 1
    return {
        "value": value,
        "m": case.sol.m[t].copy(),
        "c": case.sol.c[t].copy(),
        "age": np.arange(case.par.simT) + case.par.age_min + 1,
        "mean_assets": np.mean(case.sim.A, axis=1),
    }


experiments = {
    r"Patience $\beta$": ("beta", [0.94, 0.96, 0.98]),
    r"Risk aversion $\rho$": ("rho", [1.5, 2.0, 4.0]),
    r"Permanent risk $\sigma_\psi$": ("sigma_psi", [0.0, 0.10, 0.20]),
}

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for col, (title, (parameter, values)) in enumerate(experiments.items()):
    for value in values:
        result = solve_buffer_case(parameter, value)
        label = f"{parameter}={value:g}"
        axes[0, col].plot(result["m"], result["c"], label=label)
        axes[1, col].plot(result["age"], result["mean_assets"], label=label)

    axes[0, col].set(title=title, xlabel=r"$m_t$", ylabel=r"$c_t$", xlim=(0, 5))
    axes[1, col].set(xlabel="Age", ylabel=r"Mean assets $A_t$")
    axes[0, col].legend()
    axes[1, col].legend()

fig.suptitle("One-at-a-time comparative statics", y=1.02)
fig.tight_layout()
plt.show()
